In [1]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "공부가 너무 손에 안 잡혀요. 어떻게 해야 할까요?",
    "요즘 취업 걱정 때문에 밤에 잠을 잘 수가 없습니다.",
    "친구랑 심하게 다퉜는데 화해하는 방법을 모르겠어.",
    "운동을 시작했는데 살이 전혀 안 빠져서 속상합니다."
]

okt = Okt()
stop_words = ['너무', '어떻게', '때문에', '잘', '수가', '전혀', '은', '는', '이', '가', '을', '를', '하다', '합니다', '에', '에서', '요', '아', '휴']

def tokenizer_with_stopwords(text):
    tokens = okt.morphs(text, stem=True)
    
    clean_tokens = [token for token in tokens if token not in stop_words]
    return clean_tokens
  
tfidf_vectorizer = TfidfVectorizer(tokenizer=tokenizer_with_stopwords, token_pattern=None)

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

print("=== 전처리 및 벡터화 결과 ===")
print(f"희소 행렬의 크기(Shape): {tfidf_matrix.shape}")
print(f"  -> 설명: 총 {tfidf_matrix.shape[0]}개의 문장(행)과 {tfidf_matrix.shape[1]}개의 고유 단어(열)로 구성됨\n")

print("[추출된 고유 단어 집합 (Vocabulary)]")
print(tfidf_vectorizer.get_feature_names_out())

=== 전처리 및 벡터화 결과 ===
희소 행렬의 크기(Shape): (4, 27)
  -> 설명: 총 4개의 문장(행)과 27개의 고유 단어(열)로 구성됨

[추출된 고유 단어 집합 (Vocabulary)]
['.' '?' '걱정' '공부' '다투다' '때문' '랑' '모르다' '밤' '방법' '빠지다' '살이' '속상하다' '손'
 '시작' '심하다' '안' '어떻다' '없다' '요즘' '운동' '자다' '잠' '잡히다' '취업' '친구' '화해']


In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

labels = np.array([0, 0, 0, 1])

X_train, X_test, y_train, y_test = train_test_split(
    tfidf_matrix, labels, test_size=0.25, random_state=42
)

model = LogisticRegression()

model.fit(X_train, y_train)

print("=== 모델 학습 완료 ===")
print(f"학습 데이터 개수: {X_train.shape[0]}개, 테스트 데이터 개수: {X_test.shape[0]}개\n")

test_sentences = [
    "요즘 운동을 시작했더니 기분이 정말 상쾌해지고 좋아졌어!",
    "시험 준비하느라 스트레스 받아서 매일 우울하고 힘들어."
]

test_matrix = tfidf_vectorizer.transform(test_sentences)

predictions = model.predict(test_matrix)
probabilities = model.predict_proba(test_matrix)

print("=== 감성 분석 테스트 결과 ===")
for i, sentence in enumerate(test_sentences):
    sentiment = "긍정" if predictions[i] == 1 else "부정"
    prob = probabilities[i][predictions[i]] * 100
    
    print(f"[{i+1}] 문장: '{sentence}'")
    print(f" -> 예측 감성: {sentiment}")
    print(f" -> 예측 확률: {prob:.2f}%\n")

=== 모델 학습 완료 ===
학습 데이터 개수: 3개, 테스트 데이터 개수: 1개

=== 감성 분석 테스트 결과 ===
[1] 문장: '요즘 운동을 시작했더니 기분이 정말 상쾌해지고 좋아졌어!'
 -> 예측 감성: 부정
 -> 예측 확률: 61.25%

[2] 문장: '시험 준비하느라 스트레스 받아서 매일 우울하고 힘들어.'
 -> 예측 감성: 부정
 -> 예측 확률: 67.17%



In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.svm import SVC
data = {
    'content': [
        "공부가 너무 안 됩니다. 미치겠어요.",
        "취업 준비로 스트레스가 많아요.",
        "너무 우울하고 힘든 하루였어.",
        "계속 실패만 해서 좌절감이 듭니다.",
        "아무것도 하기 싫고 무기력해요.",
        "운동을 하니 상쾌하고 기분이 좋아!",
        "오늘 날씨가 좋아서 기분이 최고입니다.",
        "친구와 화해해서 정말 다행이야.",
        "원하던 회사에 합격해서 날아갈 것 같아요!",
        "맛있는 걸 먹었더니 스트레스가 다 풀리네요."
    ],
    'label': [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
}
df = pd.DataFrame(data)
df_filtered = df[df['content'].str.len() > 5].copy()
print(f"5자 이하 삭제 전: {len(df)}개 -> 삭제 후: {len(df_filtered)}개\n")
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df_filtered['content'])
y = df_filtered['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
svm_model = SVC(random_state=42)
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}
kf = KFold(n_splits=3, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=svm_model,
    param_grid=param_grid,
    cv=kf,
    scoring='accuracy',
    n_jobs=-1
)
print("하이퍼파라미터 튜닝 및 교차 검증을 시작합니다...")
grid_search.fit(X_train, y_train)
print("\n=== 최적화 및 검증 결과 ===")
print(f"1. 최적의 하이퍼파라미터 조합: {grid_search.best_params_}")
print(f"2. 교차 검증 최고 정확도: {grid_search.best_score_ * 100:.2f}%")
best_model = grid_search.best_estimator_
final_accuracy = best_model.score(X_test, y_test)
print(f"3. 한 번도 보지 못한 테스트 데이터(최종) 정확도: {final_accuracy * 100:.2f}%")

5자 이하 삭제 전: 10개 -> 삭제 후: 10개

하이퍼파라미터 튜닝 및 교차 검증을 시작합니다...

=== 최적화 및 검증 결과 ===
1. 최적의 하이퍼파라미터 조합: {'C': 10, 'kernel': 'linear'}
2. 교차 검증 최고 정확도: 22.22%
3. 한 번도 보지 못한 테스트 데이터(최종) 정확도: 50.00%
